# [실습 7] 허깅페이스 한국어 데이터셋으로 LLM 강화학습

이번 실습에서는 **허깅페이스(Hugging Face)** 의 한국어 데이터셋을 활용해 작은 언어모델을 단계별로 학습시킵니다.

| 단계 | 학습 방식 | 사용 데이터셋 | 핵심 개념 |
|------|-----------|--------------|-----------|
| 1 | **SFT** (Supervised Fine-Tuning, 모방학습) | `Beomi/KoAlpaca-v1.1a` | 시범 데이터를 그대로 따라하기 |
| 2 | **Reward** (보상 모델 컨셉) | 직접 정의한 규칙 기반 보상 | `s, a → r` 매핑 |
| 3 | **DPO** (Direct Preference Optimization) | `maywell/ko_Ultrafeedback_binarized` | 선호도 비교로 정렬 |
| 4 | **PPO 미니 시뮬레이션** | 토큰 레벨 정책경사 | Reward → log π 업데이트 |

> **Colab 권장 환경**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

> ⚠️ 무료 T4에서도 돌아가도록 `Qwen2.5-0.5B` 또는 그보다 작은 모델 + **LoRA(QLoRA)** 어댑터로 진행합니다.

---
## 0. 환경 설정

Colab에서 처음 실행한다면 아래 셀로 필요한 라이브러리를 설치합니다. (이미 설치되어 있다면 스킵)

In [ ]:
# Colab 전용 설치 셀 (로컬 환경에서는 주석 처리)
!pip install -q --upgrade \
    transformers==4.44.2 \
    datasets==2.21.0 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    trl==0.10.1 \
    bitsandbytes==0.43.3 \
    sentencepiece

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # 로컬 OpenMP 충돌 방지

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

---
## 1. 한국어 데이터셋 둘러보기

허깅페이스 허브에서 자주 쓰이는 한국어 데이터셋을 직접 불러와 형태를 확인합니다.

- **KoAlpaca**: Instruction-Following 학습용 (Stanford Alpaca의 한글 버전)
- **Ko-Ultrafeedback**: DPO/RLHF용 선호도 쌍 데이터 (`chosen` / `rejected`)
- **KLUE-YNAT** *(보너스)*: 분류 데이터지만 보상 함수 만들 때 토픽 라벨로 활용 가능

In [ ]:
from datasets import load_dataset

# SFT용 한국어 지시 데이터
sft_ds = load_dataset("Beomi/KoAlpaca-v1.1a", split="train")
print("[KoAlpaca] 총 샘플:", len(sft_ds))
print("컬럼:", sft_ds.column_names)
print("\n--- 샘플 1 ---")
print("instruction:", sft_ds[0]["instruction"])
print("output     :", sft_ds[0]["output"][:200], "...")

In [ ]:
# DPO용 한국어 선호도 데이터 (chosen / rejected 쌍)
# 데이터가 크므로 일부만 스트리밍/슬라이싱
pref_ds = load_dataset("maywell/ko_Ultrafeedback_binarized", split="train[:2000]")
print("[Ko-Ultrafeedback] 샘플:", len(pref_ds))
print("컬럼:", pref_ds.column_names)

sample = pref_ds[0]
print("\n--- 샘플 1 ---")
print("prompt  :", str(sample.get("prompt", ""))[:120])
print("chosen  :", str(sample.get("chosen", ""))[:120])
print("rejected:", str(sample.get("rejected", ""))[:120])

### ⚙️ Mini Quiz
- SFT 데이터셋은 `(instruction, output)` 한 쌍 → **"답은 이거야"** 라고 알려주는 형태.
- 선호도 데이터셋은 `(prompt, chosen, rejected)` 한 묶음 → **"이 답이 저 답보다 낫다"** 는 비교 신호.

강화학습 관점에서 보면:
- SFT = **모방학습 (Imitation Learning)**
- 선호도 학습 = **보상 모델링 → 정책 최적화**

---
## 2. 1단계 — SFT (모방학습)

`Qwen2.5-0.5B-Instruct` 베이스 모델에 **LoRA 어댑터**를 붙여 KoAlpaca를 학습합니다.  
Colab 무료 T4 기준 100스텝 정도면 충분히 응답 톤이 한국어로 정렬됩니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# T4(16GB)에서도 안정적으로 돌아가도록 4-bit 양자화 (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print("베이스 모델 로드 완료.")

In [ ]:
# 프롬프트 포맷팅: KoAlpaca → Alpaca 스타일
def format_alpaca(example):
    texts = []
    for inst, out in zip(example["instruction"], example["output"]):
        texts.append(
            f"### 명령어:\n{inst}\n\n### 응답:\n{out}{tokenizer.eos_token}"
        )
    return texts

# 실습 속도를 위해 일부만 사용
train_subset = sft_ds.shuffle(seed=42).select(range(1000))

peft_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

training_args = TrainingArguments(
    output_dir="./sft_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=100,           # 데모용 — 실제 학습에서는 ↑
    logging_steps=10,
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="constant",
    warmup_ratio=0.03,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_subset,
    peft_config=peft_config,
    formatting_func=format_alpaca,
    tokenizer=tokenizer,
    max_seq_length=512,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained("./sft_lora_adapter")
print("SFT 완료 — ./sft_lora_adapter 에 어댑터 저장.")

In [ ]:
# 학습된 모델로 한국어 응답 생성 테스트
def generate(prompt, max_new_tokens=128):
    text = f"### 명령어:\n{prompt}\n\n### 응답:\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=True, temperature=0.7, top_p=0.9,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

for q in ["점심 메뉴 추천해줘. 매운 건 못 먹어.",
         "파이썬 리스트를 역순으로 뒤집는 방법은?",
         "과적합을 막는 방법 세 가지를 알려줘."]:
    print("Q:", q)
    print("A:", generate(q))
    print("-" * 60)

---
## 3. 2단계 — 보상 함수 정의 (Reward 컨셉)

RLHF에서는 보통 **사람 피드백 → 보상모델(Reward Model)** 을 학습하지만,  
이 실습에서는 강화학습의 **"보상 신호"** 가 어떤 역할을 하는지 직관적으로 보기 위해 **규칙 기반 보상**을 만들어 봅니다.

### 보상 규칙 예시
- ✅ 한글 비율이 높으면 + (한국어 응답 유도)
- ✅ 200자 이상이면 + (성의 있는 답변)
- ❌ 매운 음식 추천 키워드가 등장하면 − (사용자 제약 위반)
- ❌ 응답에 영어 비속어 / 'sorry I cannot' 류가 있으면 −

In [ ]:
import re

SPICY = ["매운", "불닭", "떡볶이", "마라", "청양"]
REFUSAL = ["sorry i cannot", "i can't help", "as an ai"]

def korean_ratio(text: str) -> float:
    if not text:
        return 0.0
    kor = len(re.findall(r"[가-힣]", text))
    return kor / max(len(text), 1)

def reward_fn(prompt: str, response: str) -> float:
    r = 0.0
    # 한국어 비율 보상
    r += 2.0 * korean_ratio(response)
    # 길이 보상 (너무 짧지도 길지도 않게)
    L = len(response)
    r += 1.0 if 80 <= L <= 400 else -0.5
    # 제약 위반 페널티: 사용자가 '매운 거 못 먹어'라고 했는데 매운 음식을 추천하면 -
    if "매운" in prompt and any(w in response for w in SPICY):
        r -= 3.0
    # 거절 톤 페널티
    low = response.lower()
    if any(w in low for w in REFUSAL):
        r -= 2.0
    return r

# 데모
print(reward_fn("매운 건 못 먹어. 점심 추천", "매운 떡볶이가 최고죠!"))    # 페널티
print(reward_fn("매운 건 못 먹어. 점심 추천", "담백한 돈가스나 갈비탕이 좋습니다."))  # 보상

---
## 4. 3단계 — DPO (Direct Preference Optimization)

**DPO** 는 PPO처럼 보상모델을 따로 두지 않고, **선호도 쌍 (chosen, rejected)** 만으로 정책을 직접 정렬합니다.

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

직관적으로 "**좋은 답변의 확률은 ↑, 나쁜 답변의 확률은 ↓**" 를 한 번에 학습합니다.

In [ ]:
from trl import DPOTrainer
from transformers import TrainingArguments

# Ko-Ultrafeedback → TRL이 기대하는 (prompt, chosen, rejected) 컬럼 정리
def to_dpo_format(ex):
    return {
        "prompt":   str(ex.get("prompt", "")).strip(),
        "chosen":   str(ex.get("chosen", "")).strip(),
        "rejected": str(ex.get("rejected", "")).strip(),
    }

dpo_train = pref_ds.map(to_dpo_format,
                        remove_columns=[c for c in pref_ds.column_names
                                        if c not in ("prompt", "chosen", "rejected")])
dpo_train = dpo_train.filter(lambda x: len(x["prompt"]) > 0
                                       and len(x["chosen"]) > 0
                                       and len(x["rejected"]) > 0)
dpo_train = dpo_train.select(range(min(500, len(dpo_train))))  # 데모용 축소
print("DPO 학습 샘플:", len(dpo_train))
print(dpo_train[0])

In [ ]:
dpo_args = TrainingArguments(
    output_dir="./dpo_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=80,            # 데모 — 실제는 더 길게
    logging_steps=10,
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none",
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=model,            # SFT 어댑터가 붙은 모델을 이어서 사용
    ref_model=None,         # PEFT의 경우 자동으로 ref를 분리 처리
    args=dpo_args,
    beta=0.1,
    train_dataset=dpo_train,
    tokenizer=tokenizer,
    max_length=512,
    max_prompt_length=256,
)

dpo_trainer.train()
dpo_trainer.model.save_pretrained("./dpo_lora_adapter")
print("DPO 완료 — ./dpo_lora_adapter 저장.")

In [ ]:
# DPO 이후 응답이 어떻게 달라졌는지 확인
for q in ["매운 음식 못 먹는데 점심 메뉴 추천해줘.",
         "머신러닝에서 과적합을 막는 방법 세 가지만 말해줘."]:
    ans = generate(q)
    print("Q:", q)
    print("A:", ans)
    print(f"   ↳ reward = {reward_fn(q, ans):.3f}")
    print("-" * 60)

---
## 5. 4단계 — PPO 미니 시뮬레이션 (정책경사 직관 잡기)

실제 PPO는 큰 LLM에서 돌리기 무겁기 때문에, 여기서는 **장난감 언어모델 + 토큰 단위 정책경사** 로  
"보상 → log 확률 업데이트" 의 메커니즘을 손으로 확인합니다.

(`rein_test.py` 를 노트북 흐름에 맞게 정리한 버전입니다.)

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

VOCAB = 10        # 단어 사전 크기
GOAL_TOKEN = 5    # 이 토큰이 들어가야 "좋은 답변"

class TinyLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, 16)
        self.lstm = nn.LSTM(16, 32, batch_first=True)
        self.head = nn.Linear(32, VOCAB)
    def forward(self, x):
        h, _ = self.lstm(self.emb(x))
        return self.head(h)

def toy_reward(seq):
    return 10.0 if GOAL_TOKEN in seq else -2.0

torch.manual_seed(0)
policy = TinyLM()
opt = optim.Adam(policy.parameters(), lr=1e-2)
prompt = torch.tensor([[1, 2, 3]])

rewards_log = []
for ep in range(50):
    opt.zero_grad()
    logits = policy(prompt)
    probs = torch.softmax(logits, dim=-1)

    tokens, log_probs = [], []
    for t in range(3):
        dist = Categorical(probs[0, t])
        a = dist.sample()
        tokens.append(a.item())
        log_probs.append(dist.log_prob(a))

    r = toy_reward(tokens)
    loss = -torch.stack(log_probs).sum() * r     # REINFORCE
    loss.backward()
    opt.step()
    rewards_log.append(r)
    if (ep + 1) % 10 == 0:
        print(f"Ep {ep+1:>2} | tokens={tokens} | r={r:+.1f} | loss={loss.item():+.3f}")

print(f"\n최근 10 에피소드 평균 보상: {sum(rewards_log[-10:])/10:.2f}")

In [ ]:
# 보상 곡선 시각화 — 처음에는 음수가 많다가, 점차 +10에 수렴
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(rewards_log, marker="o", linewidth=1)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.7)
plt.title("Toy Policy Gradient — Reward per Episode")
plt.xlabel("episode"); plt.ylabel("reward")
plt.tight_layout(); plt.show()

---
## 6. 정리 & 응용 과제

### 오늘 익힌 것
1. 허깅페이스 한국어 데이터셋 로딩 — `KoAlpaca`, `Ko-Ultrafeedback`
2. **SFT** = 시범 데이터를 정답으로 두는 모방학습
3. **보상 함수** = 상태·행동(생성문)에 점수를 매기는 신호
4. **DPO** = 보상모델 없이 선호도 쌍으로 정책을 정렬
5. **REINFORCE** = `loss = -log π(a|s) * R` — RLHF의 PPO도 같은 뿌리

### 응용 과제 (택 1)
- 🔧 **A. 보상 다양화**: `reward_fn`에 "존댓말 사용 시 +", "숫자/팩트가 포함되면 +" 같은 규칙을 추가하고 DPO를 다시 돌렸을 때 응답이 어떻게 바뀌는지 비교해 보세요.
- 🔧 **B. 데이터셋 교체**: `HAERAE-HUB/HAE-RAE-Bench` 또는 `nlpai-lab/kullm-v2` 로 SFT 데이터를 바꿔보고, 도메인이 응답 스타일에 미치는 영향을 관찰해 보세요.
- 🔧 **C. PPO 본격 도입**: `trl.PPOTrainer` 로 위 `reward_fn` 을 그대로 보상으로 연결해 LLM을 강화학습으로 미세조정해 보세요.

### 참고 링크
- KoAlpaca: <https://huggingface.co/datasets/Beomi/KoAlpaca-v1.1a>
- Ko-Ultrafeedback: <https://huggingface.co/datasets/maywell/ko_Ultrafeedback_binarized>
- TRL 문서 (SFT/DPO/PPO): <https://huggingface.co/docs/trl>
